# 🏗️ ITF Agentic Platform - Chapter 2: Architecture Deep Dive

**Learning Goals:**
- Understand each architectural layer
- See code examples
- Make actual API calls
- Understand data flow

**Last Updated:** 2026-10-24

---

## Setup: Import Libraries & Configure API

In [ ]:
import requests
import json
import boto3
from decimal import Decimal
from datetime import datetime
import time

# Configuration
API_BASE_URL = "http://localhost:8080"
DYNAMODB_ENDPOINT = "http://localhost:21003"
DYNAMODB_REGION = "us-east-1"

print("✓ Libraries imported")
print(f"✓ API URL: {API_BASE_URL}")
print(f"✓ DynamoDB: {DYNAMODB_ENDPOINT}")

---

## Layer 1: REST API Layer

### What it does:
Receives HTTP requests and routes them to handlers

**File:** `/app/api/agent.py`

**Endpoints:**
```python
@router.get("/agent/oob/agents")                    # List agents
@router.post("/agent/invoke/{agent_id}")            # Invoke agent
@router.post("/agent/stream/{agent_id}")            # Stream response
@router.get("/agent/tokens/{agent_id}")             # Get token count
```

### Test: List Available Agents

In [ ]:
# Test Layer 1: REST API
print("🔍 Testing REST API Layer...\n")

response = requests.get(f"{API_BASE_URL}/agent/oob/agents")

print(f"Status Code: {response.status_code}")
print(f"Response:\n")

data = response.json()
print(json.dumps(data, indent=2))

# Extract agents
agents = data.get('agents', [])
print(f"\n✓ Found {len(agents)} agent(s)")
for agent in agents:
    print(f"  - ID: {agent['agent_id']}")
    print(f"    Name: {agent['name']}")
    print(f"    Description: {agent['description'][:50]}...")

---

## Layer 2: Agent Factory & Loader

### What it does:
1. Takes agent_id
2. Queries DynamoDB for config
3. Creates AgentCard object
4. Returns ready-to-use agent

**File:** `/app/core/agent_factory.py`

**Key Function:**
```python
def load_agent_by_id(agent_id: str) -> AgentCard:
    # Query DynamoDB
    config = get_agent_config(agent_id)
    # Create agent object
    agent = AgentCard(**config)
    return agent
```

### Demo: Connect to DynamoDB and Load Agent

In [ ]:
print("🔍 Testing Agent Factory Layer...\n")

# Connect to DynamoDB
dynamodb = boto3.resource(
    'dynamodb',
    region_name=DYNAMODB_REGION,
    endpoint_url=DYNAMODB_ENDPOINT
)

print("✓ Connected to DynamoDB\n")

# Get table
table = dynamodb.Table('itf-agent-configuration-us-east-1-dev-01')

print("✓ Loaded Agent Configuration Table\n")

# Query for agent
print("Querying for agents...\n")
response = table.scan()

items = response.get('Items', [])
print(f"Found {len(items)} agent(s) in database\n")

for item in items:
    print(f"Agent ID: {item.get('agent_id')}")
    print(f"Name: {item.get('name')}")
    print(f"LLM Model: {item.get('llm_model')}")
    print(f"Tools: {item.get('tools', [])}")
    print(f"Temperature: {item.get('temperature')}")
    print(f"Max Tokens: {item.get('max_tokens')}")
    print()

---

## Layer 3: Conversation Service

### What it does:
1. Saves conversation history
2. Loads previous messages for context
3. Manages sessions

**File:** `/app/services/conversation_service.py`

**Key Functions:**
```python
def get_conversation_history(session_id: str) -> List[Message]:
    # Query DynamoDB for all messages in this session
    return messages

def save_message(session_id: str, message: Message) -> None:
    # Save message to DynamoDB
    pass
```

### Demo: Check Conversation History Table

In [ ]:
print("🔍 Testing Conversation Service Layer...\n")

# Get conversation history table
conv_table = dynamodb.Table('itf-conversation-history-us-east-1-dev-01')

print("✓ Loaded Conversation History Table\n")

# Query for conversations
response = conv_table.scan()
items = response.get('Items', [])

print(f"Total conversations: {len(items)}\n")

if items:
    for item in items:
        print(f"Session ID: {item.get('PK')}")
        print(f"Messages: {item.get('messages', [])}")
        print()
else:
    print("✓ No conversations yet (table is empty)")
    print("  → Conversations will be saved here after you invoke agents")

---

## Layer 4: Agent Executor

### What it does:
1. Takes agent + user input
2. Loads conversation history
3. Creates tools
4. Calls LLM (Claude)
5. Handles tool calls
6. Returns response

**File:** `/app/core/agent_executor.py`

**Key Function:**
```python
async def execute_agent(
    agent: AgentCard,
    request: AgentRequest
) -> AgentResponse:
    # Load history
    history = get_conversation_history(request.session_id)
    # Create tools
    tools = create_mcp_tools(agent.tools)
    # Call LLM
    response = call_llm(agent, history, tools)
    # Save history
    save_messages(request.session_id, response)
    return response
```

### Demo: Invoke Agent

In [ ]:
print("🔍 Testing Agent Executor Layer...\n")

# Prepare request
agent_id = "agent-sample"
session_id = f"test-session-{int(time.time())}"
user_input = "Say hello and tell me what you can do"

print(f"Agent ID: {agent_id}")
print(f"Session ID: {session_id}")
print(f"User Input: {user_input}\n")

# Make request
payload = {
    "user_input": user_input,
    "session_id": session_id
}

print("Sending request to agent...\n")

try:
    response = requests.post(
        f"{API_BASE_URL}/agent/invoke/{agent_id}",
        json=payload,
        timeout=30
    )
    
    print(f"Status Code: {response.status_code}\n")
    
    data = response.json()
    print("Response from Agent:")
    print(json.dumps(data, indent=2))
    
except requests.exceptions.Timeout:
    print("⚠️  Request timed out - agent might need external tools")
except Exception as e:
    print(f"❌ Error: {e}")

---

## Complete Data Flow Visualization

```
USER REQUEST
    ↓
┌─────────────────────────────────────┐
│ LAYER 1: REST API                   │
│ POST /agent/invoke/agent-sample     │
│ {                                   │
│   "user_input": "...",             │
│   "session_id": "test-001"         │
│ }                                   │
└────────────┬────────────────────────┘
             ↓
┌──────────────────────────────────────┐
│ LAYER 2: AGENT FACTORY               │
│ load_agent_by_id("agent-sample")    │
│ └─ Query DynamoDB                    │
│ └─ Create AgentCard object           │
│ └─ Return agent config               │
└────────────┬─────────────────────────┘
             ↓
┌──────────────────────────────────────┐
│ LAYER 3: CONVERSATION SERVICE        │
│ get_conversation_history(session_id) │
│ └─ Query conversation history table  │
│ └─ Load previous messages            │
└────────────┬─────────────────────────┘
             ↓
┌──────────────────────────────────────┐
│ LAYER 4: TOOL FACTORY                │
│ create_mcp_tools(agent.tools)        │
│ └─ Connect to MCP servers            │
│ └─ Fetch tool definitions            │
└────────────┬─────────────────────────┘
             ↓
┌──────────────────────────────────────┐
│ LAYER 5: AGENT EXECUTOR              │
│ execute_agent(agent, request)        │
│ └─ Prepare context                   │
│ └─ Call Claude LLM                   │
│ └─ Handle tool calls (if any)        │
└────────────┬─────────────────────────┘
             ↓
┌──────────────────────────────────────┐
│ LAYER 6: PERSISTENCE                 │
│ save_message(session_id, response)   │
│ └─ Save to conversation history      │
│ └─ Store in DynamoDB                 │
└────────────┬─────────────────────────┘
             ↓
┌──────────────────────────────────────┐
│ RESPONSE                             │
│ {                                    │
│   "response": "...",                │
│   "status": "success",              │
│   "session_id": "test-001"          │
│ }                                    │
└──────────────────────────────────────┘
```

---

## Architecture Summary

In [ ]:
summary = {
    "Layer 1: REST API": {
        "Purpose": "Receive HTTP requests",
        "File": "app/api/agent.py",
        "Key Function": "invoke_agent(agent_id, request)"
    },
    "Layer 2: Agent Factory": {
        "Purpose": "Load agent config from database",
        "File": "app/core/agent_factory.py",
        "Key Function": "load_agent_by_id(agent_id)"
    },
    "Layer 3: Conversation Service": {
        "Purpose": "Manage conversation history",
        "File": "app/services/conversation_service.py",
        "Key Functions": "get_conversation_history(), save_message()"
    },
    "Layer 4: Tool Factory": {
        "Purpose": "Create and manage tools",
        "File": "app/core/tool_factory.py",
        "Key Function": "create_mcp_tools(tools_config)"
    },
    "Layer 5: Agent Executor": {
        "Purpose": "Run agent and call LLM",
        "File": "app/core/agent_executor.py",
        "Key Function": "execute_agent(agent, request)"
    },
    "Layer 6: Persistence": {
        "Purpose": "Store data in database",
        "Database": "DynamoDB",
        "Tables": [
            "itf-agent-configuration-us-east-1-dev-01",
            "itf-conversation-history-us-east-1-dev-01"
        ]
    }
}

print("📋 ARCHITECTURE LAYERS:\n")
for layer, details in summary.items():
    print(f"{layer}")
    for key, value in details.items():
        if isinstance(value, list):
            print(f"  {key}:")
            for item in value:
                print(f"    - {item}")
        else:
            print(f"  {key}: {value}")
    print()

---

## Key Takeaways

✓ **Modular Architecture** - Each layer has single responsibility

✓ **Data Flow** - Clear path from request to response

✓ **Persistence** - Agents and conversations stored in DynamoDB

✓ **Scalability** - Each layer can be scaled independently

✓ **Flexibility** - Easy to add new agents or tools

## What's Next?

**Chapter 3:** Deep dive into Agent Factory

**Chapter 4:** Understanding Tool Integration (MCP)

**Chapter 5:** Building Your First Custom Agent